# 02 — Variable categorization

**Input:** `outputs/dataset_clean.csv` (output from Notebook 01)  
**Output:** `outputs/dataset_categorized.csv`

Adds five categorical factors: age group, sex category, systolic blood pressure category, diastolic blood pressure category, and BMI category. **No participant is excluded**, including participants younger than 60 years. All original columns and numeric measurements remain unchanged.

### Operational definitions

- **Age:** `<60`, `60–69`, `70–79`, `80+` (unchanged, study-defined intervals).
- **Sex:** original coding `1 = Female`, `2 = Male` (unchanged).
- **Systolic blood pressure:** two measurement-defined groups, `<140 mmHg` and `≥140 mmHg`.
- **Diastolic blood pressure:** two measurement-defined groups, `<90 mmHg` and `≥90 mmHg`.
- **BMI:** MINSAL general-adult thresholds for ages `<60`; MINSAL older-person thresholds for ages `≥60` (unchanged). The selected reference is stored in `bmi_reference`.

The 140 mmHg systolic and 90 mmHg diastolic cutoffs were already used in the original notebook as the start of the former Stage 1 level. Here, they are applied **independently to each blood-pressure measurement**; neither category represents a confirmed hypertension diagnosis. The former combined `bp_category` is no longer generated.

**References retained from the original notebook:**

1. MINSAL (2021), *Plan Nacional de Salud Integral para Personas Mayores*, Table 19: https://www.minsal.cl/wp-content/uploads/2021/11/Plan-Nacional-de-Salud-Integral-para-Personas-Mayores.pdf
2. MINSAL, *Guía Clínica 2010 — Hipertensión arterial primaria*, Table 2: https://www.minsal.cl/portal/url/item/7220fdc4341c44a9e04001011f0113b9.pdf
3. MINSAL, *Guía del Examen de Medicina Preventiva*, adult BMI Table 1: https://diprece.minsal.cl/wrdprss_minsal/wp-content/uploads/2014/09/GPC-Medicina-Preventiva.pdf
4. Chile, Decreto 20 (2022), definition of *persona mayor*: https://www.leychile.cl/navegar?idNorma=1182129

No transformations, outlier treatment, HRV calculations, or analytical-cohort filtering are performed here.

In [1]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../../outputs/dataset_clean.csv")
OUTPUT_FILE = Path("../../outputs/dataset_categorized.csv")

df = pd.read_csv(INPUT_FILE)
original_columns = df.columns.tolist()
original_rows = len(df)

required = ["source_excel_row", "age", "sex", "bp_systolic", "bp_diastolic", "imc"]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Input: {INPUT_FILE} | Participants: {original_rows} | Columns: {len(original_columns)}")


Input: ../../outputs/dataset_clean.csv | Participants: 530 | Columns: 19


In [2]:
# Study-defined age bands; no age-based filtering.
df["age_group"] = pd.cut(
    df["age"],
    bins=[float("-inf"), 60, 70, 80, float("inf")],
    right=False,
    labels=["<60", "60-69", "70-79", "80+"],
).astype("string")

# Original sex coding: 1=Female and 2=Male (verify if the REDCap dictionary differs).
df["sex_category"] = df["sex"].map({1: "Female", 2: "Male"}).astype("string")

# Independently categorize systolic and diastolic BP using the existing 140/90 mmHg cutoffs.
# Categories are measurement descriptions, not diagnoses. Missing/non-positive values stay unassigned.
sbp = df["bp_systolic"]
dbp = df["bp_diastolic"]

sbp_valid = sbp.notna() & (sbp > 0)
dbp_valid = dbp.notna() & (dbp > 0)

df["sbp_category"] = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[sbp_valid & (sbp < 140), "sbp_category"] = "<140 mmHg"
df.loc[sbp_valid & (sbp >= 140), "sbp_category"] = ">=140 mmHg"

df["dbp_category"] = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[dbp_valid & (dbp < 90), "dbp_category"] = "<90 mmHg"
df.loc[dbp_valid & (dbp >= 90), "dbp_category"] = ">=90 mmHg"

In [3]:
# Age-specific MINSAL BMI categories; numeric BMI is not changed.
# Adults (<60): underweight <18.5, normal [18.5,25), overweight [25,30), obese >=30.
# Older persons (>=60): underweight <23, normal [23,28), overweight [28,32), obese >=32.

adult_bmi = pd.cut(
    df["imc"],
    bins=[0, 18.5, 25, 30, float("inf")],
    right=False,
    labels=["Underweight", "Normal", "Overweight", "Obesity"],
).astype("string")
older_bmi = pd.cut(
    df["imc"],
    bins=[0, 23, 28, 32, float("inf")],
    right=False,
    labels=["Underweight", "Normal", "Overweight", "Obesity"],
).astype("string")

df["bmi_category"] = adult_bmi
df["bmi_reference"] = "MINSAL adult (<60)"
older = df["age"] >= 60
df.loc[older, "bmi_category"] = older_bmi.loc[older]
df.loc[older, "bmi_reference"] = "MINSAL older person (>=60)"
df.loc[df["age"].isna(), ["bmi_category", "bmi_reference"]] = pd.NA


In [4]:
# Review category sizes and any unassigned cases.
factors = ["age_group", "sex_category", "sbp_category", "dbp_category", "bmi_category"]
for factor in factors:
    print(f"\n{factor}")
    display(df[factor].value_counts(dropna=False).rename("n").to_frame())

print("\nBMI classification reference")
display(df["bmi_reference"].value_counts(dropna=False).rename("n").to_frame())

print("\nUnassigned categories (if any)")
display(df[factors].isna().sum().rename("missing_n").to_frame())


age_group


,n
age_group,
70-79,261
60-69,219
80+,40
<60,10



sex_category


,n
sex_category,
Female,366
Male,164



sbp_category


,n
sbp_category,
<140 mmHg,395
>=140 mmHg,135


... (4 outputs left out)


BMI classification reference


,n
bmi_reference,
MINSAL older person (>=60),520
MINSAL adult (<60),10



Unassigned categories (if any)


,missing_n
age_group,0
sex_category,0
sbp_category,0
dbp_category,0
bmi_category,0


In [5]:
# Export the categorized dataset without changing or filtering input data.
assert len(df) == original_rows
assert df.columns[:len(original_columns)].tolist() == original_columns
assert df["source_excel_row"].is_unique
assert "bp_category" not in df.columns
assert all(factor in df.columns for factor in factors)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE} | {len(df)} participants | {df.shape[1]} columns")
display(df[["source_excel_row", "age", "age_group", "sex", "sex_category",
            "bp_systolic", "sbp_category", "bp_diastolic", "dbp_category",
            "imc", "bmi_category", "bmi_reference"]].head())

Saved: ../../outputs/dataset_categorized.csv | 530 participants | 25 columns


,source_excel_row,age,age_group,sex,sex_category,bp_systolic,sbp_category,bp_diastolic,dbp_category,imc,bmi_category,bmi_reference
0,2,75,70-79,1,Female,108,<140 mmHg,70,<90 mmHg,27.343750,Normal,MINSAL older person (>=60)
1,3,66,60-69,1,Female,122,<140 mmHg,72,<90 mmHg,23.422368,Normal,MINSAL older person (>=60)
2,4,77,70-79,2,Male,125,<140 mmHg,85,<90 mmHg,28.027682,Overweight,MINSAL older person (>=60)
3,5,77,70-79,1,Female,126,<140 mmHg,72,<90 mmHg,33.203125,Obesity,MINSAL older person (>=60)
4,6,73,70-79,1,Female,130,<140 mmHg,80,<90 mmHg,30.261831,Overweight,MINSAL older person (>=60)
